In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Work preference survey

Quick reminder — `pd.crosstab`:
```python
pd.crosstab(df['rows'], df['cols'])                          # counts
pd.crosstab(df['rows'], df['cols'], margins=True)            # with row/col totals
pd.crosstab(df['rows'], df['cols'],
            values=df['val'], aggfunc='mean')                # aggregate a value
```

- Build a crosstab of `dept` × `preference`. Add `margins=True`. Which department has the most Remote preferences?
- Build a second crosstab using `values=survey['satisfied']` and `aggfunc='mean'`. Which dept × preference cell has the highest average satisfaction?
- Use `np.nanmean` to compute overall average satisfaction across the whole dataset.

In [12]:
survey = pd.DataFrame({
    'dept':      ['Engineering','Marketing','Engineering','Sales','Marketing',
                  'Engineering','Sales','Marketing','Engineering','Sales',
                  'Marketing','Engineering','Sales','Engineering','Marketing'],
    'seniority': ['Senior','Junior','Junior','Senior','Senior',
                  'Senior','Junior','Junior','Senior','Senior',
                  'Junior','Junior','Senior','Junior','Senior'],
    'preference':['Remote','Office','Remote','Hybrid','Remote',
                  'Hybrid','Remote','Office','Remote','Hybrid',
                  'Remote','Hybrid','Remote','Office','Hybrid'],
    'satisfied': [True, False, True, True, True,
                  True, True, False, True, False,
                  True, True, True, False, True],
})

# Your code here

c = pd.crosstab(survey['dept'],survey['preference'], margins = True)
ci = pd.crosstab(survey['dept'],survey['preference'])
print(ci['Remote'].idxmax(),'has the highest remote workers')

c2 = pd.crosstab(survey['dept'],survey['preference'],
                 values = survey['satisfied'], aggfunc='mean')
print(c2.unstack().idxmax(),'has the highest satification rate')

print(np.nanmean(survey['satisfied']))


Engineering has the highest remote workers
('Hybrid', 'Engineering') has the highest satification rate
0.7333333333333333


---

## Level 2 — Quarterly sales (wide → long)

Quick reminder:
```python
long = df.set_index(['a', 'b']).stack()       # wide → long; quarter col becomes a new index level
long.index.names = ['a', 'b', 'quarter']      # rename the new level
long.xs('Q1', level='quarter')                # filter to one level value
long.unstack('quarter')                       # long → wide (restore original shape)
```

1. Set index to `['rep', 'region']` and `stack()`. Name the resulting index levels `['rep', 'region', 'quarter']`. Name the Series `'sales'`.
2. Which quarter had the highest average sales across all reps?
3. Use `.xs()` to extract Q3 only. Which rep had the best Q3?
4. Unstack back to wide format. Add a `total` column summing all four quarters. Use `np.argmax` to find the row index of the top earner overall.

In [41]:
performance = pd.DataFrame({
    'rep':    ['Alice','Bob','Carol','Dave'],
    'region': ['North','South','North','South'],
    'Q1':     [42000, 35000, 48000, 31000],
    'Q2':     [45000, 38000, 51000, 29000],
    'Q3':     [49000, 41000, 53000, 35000],
    'Q4':     [55000, 44000, 58000, 40000],
})

# Your code here

long = performance.set_index(['rep','region']).stack()
long.index.names = ['rep','region','quarter']
long.name = 'sales'

g = long.to_frame().groupby(level = 'quarter')['sales'].mean()
print(g.idxmax(),'has the highest sales')

q3 = long.xs('Q3', level = 'quarter')
print(q3.idxmax()[0],'had the best Q3')
w = long.unstack()
w['total'] = w.sum(axis =1)
print(w.index[np.argmax(w['total'])],'is the top earner overall')


Q4 has the highest sales
Carol had the best Q3
('Carol', 'North') is the top earner overall


---

## Level 3 — Daily stock prices

Quick reminder — time series operations from Week 7:
```python
df['prev'] = df['price'].shift(1)                    # lag by 1 row
df['date'] + pd.DateOffset(weeks=2)                  # shift dates forward
df['date'].dt.isocalendar().week                     # extract ISO week number
```

No steps. Answer these:

1. Add `prev_close` (yesterday's price via `.shift(1)`) and `daily_return` (`(price - prev_close) / prev_close * 100`).
2. Which ISO week had the highest average daily return? Use `.dt.isocalendar().week` to group.
3. Add a `settlement_date` column: each trading date plus 2 business days using `pd.DateOffset(days=2)`.
4. Add `smoothed_price` using `ewm(span=5).mean()`. On which date did it first exceed 165?
5. Is there a correlation between `volume` and `daily_return`? Use `np.corrcoef` (drop the first row where `daily_return` is NaN).

In [64]:
prices = pd.DataFrame({
    'date':   pd.date_range('2024-01-02', periods=20, freq='B'),
    'price':  [150, 152, 149, 153, 155, 158, 156, 160, 162, 158,
               165, 163, 167, 170, 168, 172, 175, 173, 178, 182],
    'volume': [2100, 1850, 2300, 1950, 2400, 2800, 2200, 3100, 2950, 2600,
               3200, 2900, 3400, 3800, 3100, 3600, 4100, 3700, 4300, 4800],
})

# Your code here

prices['prev_close'] = prices['price'].shift(1)
prices['daily_return'] = (prices['price'] - prices['prev_close'])/prices['prev_close']*100

prices['iso_week'] = prices['date'].dt.isocalendar().week
pn = prices.dropna()
print(pn.groupby('iso_week')['daily_return'].mean().idxmax(),'had the highest avg return')

prices['settlement_date'] = prices['date'] + pd.DateOffset(days = 2)
prices['smoothed_price'] = prices['price'].ewm(span = 5).mean()

e = prices.loc[prices['smoothed_price']> 165]
print(e.iloc[0]['date'],'first exceeded 165')

cors = np.corrcoef(pn['volume'], pn['daily_return'])[0,1]
print('correlation is', cors)
print('mildly correlated')

5 had the highest avg return
2024-01-19 00:00:00 first exceeded 165
correlation is 0.336871029731912
mildly correlated
